In [ ]:
"""
LEAP Results dashboard workflow (utilities repo)

Notebook-first script that:
- reads exported LEAP result workbooks (transport/industry/demand_others)
- maps each sheet to 9th/ESTO sectors using config/leap_results_sheet_map.csv
- aligns fuels via config/sector_fuel_codes_to_names.xlsx + config/ninth_sector_fuel_pairs.csv
  (with optional overrides in config/leap_transport_fuel_aliases.csv)
- compares LEAP series to base-year ESTO (2022) and 9th projections
- generates charts and dashboards styled like leap_transport
"""
from __future__ import annotations

import os
import sys
from pathlib import Path
from typing import Sequence

import pandas as pd

REPO_ROOT = Path(__file__).resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utilities.leap_results_dashboard_utils import (
    DEFAULT_CODEBOOK,
    DEFAULT_FUEL_ALIASES,
    DEFAULT_NINTH_FUEL_PAIRS,
    DEFAULT_NINTH_TO_ESTO,
    DEFAULT_SHEET_MAP,
    basic_checks,
    build_sector_to_esto_flow_lookup,
    build_charts,
    build_comparisons,
    build_dashboards,
    ensure_repo_root,
    load_fuel_aliases,
    load_leap_workbook,
    load_sheet_map,
)


# -----------------------------------------------------------------------------
# Notebook-editable toggles (keep simple and explicit)
# -----------------------------------------------------------------------------
LEAP_RESULTS_DIR = REPO_ROOT / "outputs/leap_results"
# Economy/scenario filters for workbook discovery
ECONOMY_TOKEN = "USA"  # substring to match in filenames
SCENARIOS = ("Reference", "Target")
SHEET_MAP_PATH = DEFAULT_SHEET_MAP
FUEL_ALIASES_PATH = DEFAULT_FUEL_ALIASES
CODEBOOK_PATH = DEFAULT_CODEBOOK
NINTH_TO_ESTO_PATH = DEFAULT_NINTH_TO_ESTO
BASE_TABLE_PATH = REPO_ROOT / "data/00APEC_2025_low_with_subtotals.csv"
PROJECTION_TABLE_PATH = REPO_ROOT / "data/merged_file_energy_ALL_20251106.csv"
OUTPUT_DIR = REPO_ROOT / "outputs/leap_results_dashboard/USA"
BASE_YEAR = 2022  # from 00APEC_2025_low_with_subtotals.csv
PROJECTION_YEARS: Sequence[int] = tuple(range(2023, 2071))
SCENARIO_MAP = {"reference": "reference", "target": "target"}
BASE_ECONOMY = "11USA"
PROJECTION_ECONOMY = "11_USA"
CHART_BACKEND = "plotly"  # "plotly" or "static"
USE_ESTO_AGG_ONLY = True  # default: compare against ESTO aggregated via ninth_pairs_to_esto_pairs


# -----------------------------------------------------------------------------
# Core workflow
# -----------------------------------------------------------------------------
def _resolve(path: Path | str) -> Path:
    """Resolve a path relative to repo root if not absolute."""
    p = Path(path)
    return p if p.is_absolute() else (REPO_ROOT / p)


def _discover_workbooks(root: Path, economy_token: str, scenarios: Sequence[str]) -> list[Path]:
    """
    Find LEAP result workbooks in root matching economy token and scenario labels.
    """
    root = _resolve(root)
    if not root.exists():
        raise FileNotFoundError(f"LEAP results directory not found: {root}")

    economy_token = economy_token.lower()
    scen_tokens = [s.lower() for s in scenarios]
    candidates = sorted(root.glob("*.xls*"))

    matched: list[Path] = []
    for path in candidates:
        name = path.name.lower()
        if economy_token not in name:
            continue
        if any(s in name for s in scen_tokens):
            matched.append(path)
    if not matched:
        raise FileNotFoundError(
            f"No LEAP workbooks found in {root} matching economy '{economy_token}' and scenarios {scenarios}."
        )
    return matched


def run_workflow() -> dict[str, object]:
    ensure_repo_root()
    out_dir = _resolve(OUTPUT_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"[INFO] Output dir: {out_dir}")

    leap_workbooks = _discover_workbooks(LEAP_RESULTS_DIR, ECONOMY_TOKEN, SCENARIOS)
    print(f"[INFO] Using {len(leap_workbooks)} LEAP workbook(s):")
    for wb in leap_workbooks:
        print(f"  - {wb}")

    sheet_map = load_sheet_map(_resolve(SHEET_MAP_PATH))
    fuel_aliases = load_fuel_aliases(
        _resolve(FUEL_ALIASES_PATH),
        _resolve(CODEBOOK_PATH),
        _resolve(DEFAULT_NINTH_FUEL_PAIRS),
    )
    sector_flow_mapping = build_sector_to_esto_flow_lookup(_resolve(CODEBOOK_PATH))
    ninth_pairs = pd.read_excel(_resolve(NINTH_TO_ESTO_PATH))
    base_df = pd.read_csv(_resolve(BASE_TABLE_PATH))
    ninth_df = pd.read_csv(_resolve(PROJECTION_TABLE_PATH))

    # Load LEAP long data from all workbooks
    leap_frames = [load_leap_workbook(wb, sheet_map=sheet_map) for wb in leap_workbooks]
    leap_long = pd.concat(leap_frames, ignore_index=True) if leap_frames else pd.DataFrame()
    if leap_long.empty:
        raise RuntimeError("No LEAP data loaded; check workbook paths.")

    comparison_long, comparison_wide, mapping_status = build_comparisons(
        leap_long,
        sheet_map=sheet_map,
        fuel_mapping=fuel_aliases,
        sector_flow_mapping=sector_flow_mapping,
        ninth_pairs=ninth_pairs,
        base_df=base_df,
        ninth_df=ninth_df,
        base_year=BASE_YEAR,
        base_economy=BASE_ECONOMY,
        projection_economy=PROJECTION_ECONOMY,
        projection_years=PROJECTION_YEARS,
        scenario_map=SCENARIO_MAP,
        use_esto_agg_only=USE_ESTO_AGG_ONLY,
    )

    # Write outputs
    comparison_long_path = out_dir / "comparison_long.csv"
    comparison_wide_path = out_dir / "comparison_wide.csv"
    mapping_status_path = out_dir / "mapping_status.csv"
    leap_long_path = out_dir / "leap_long.csv"

    comparison_long.to_csv(comparison_long_path, index=False)
    comparison_wide.to_csv(comparison_wide_path, index=False)
    mapping_status.to_csv(mapping_status_path, index=False)
    leap_long.to_csv(leap_long_path, index=False)
    print(f"[INFO] Wrote comparison_long: {comparison_long_path}")

    # Fail fast on unmapped fuels that carry data; warn on empty unmapped rows.
    unmapped = mapping_status[~mapping_status["mapped"]]
    if not unmapped.empty:
        merged = unmapped.merge(
            leap_long,
            left_on=["sheet", "fuel_label"],
            right_on=["sheet_name", "fuel_label"],
            how="left",
            suffixes=("", "_leap"),
        )
        has_numbers = pd.to_numeric(merged["leap_value"], errors="coerce").notna()
        with_data = merged[has_numbers]
        if not with_data.empty:
            sample = with_data[["sheet", "fuel_label", "sector_code_9th"]].drop_duplicates().head(12)
            sample_table = sample.rename(
                columns={
                    "sheet": "Sheet",
                    "fuel_label": "Fuel label",
                    "sector_code_9th": "9th sector code",
                }
            ).to_string(index=False)
            unmapped_path = out_dir / "unmapped_fuels_with_data.csv"
            cols_to_save = [
                "sheet",
                "fuel_label",
                "sector_code_9th",
                "leap_value",
                "year",
                "scenario",
                "region",
            ]
            with_data[cols_to_save].to_csv(unmapped_path, index=False)
            raise RuntimeError(
                "Unmapped fuels with data detected. "
                f"Total rows: {len(with_data)}. "
                f"See details: {unmapped_path} and {mapping_status_path}. "
                "Check canonical mappings in config/sector_fuel_codes_to_names.xlsx "
                "and config/ninth_sector_fuel_pairs.csv; "
                "use config/leap_transport_fuel_aliases.csv only for explicit overrides. "
                "Also verify sector sheet mapping in config/leap_results_sheet_map.csv. "
                "Examples (first 12):\n"
                f"{sample_table}"
            )
        else:
            uniq = unmapped[["sheet", "fuel_label"]].drop_duplicates()
            print(
                f"[WARN] {len(uniq)} unmapped fuel(s) with no LEAP values. "
                f"Review {mapping_status_path} for details."
            )

    charts_dir = out_dir / "charts"
    written_charts = build_charts(comparison_long, charts_dir=charts_dir, backend=CHART_BACKEND)
    dashboard_index = build_dashboards(output_dir=out_dir, comparison_long=comparison_long, charts_dir=charts_dir)

    checks = basic_checks(sheet_map, fuel_aliases, comparison_long, mapping_status)

    return {
        "comparison_long": str(comparison_long_path),
        "comparison_wide": str(comparison_wide_path),
        "mapping_status": str(mapping_status_path),
        "leap_long": str(leap_long_path),
        "charts_written": len(written_charts),
        "dashboard_index": str(dashboard_index) if dashboard_index else None,
        "diagnostics": checks,
    }


# -----------------------------------------------------------------------------
# Bottom run block (ready for notebooks)
# -----------------------------------------------------------------------------
if __name__ == "__main__":  # pragma: no cover
    try:
        result = run_workflow()
        print("[OK] LEAP Results dashboard workflow complete.")
        for k, v in result.items():
            print(f"- {k}: {v}")
    except Exception as exc:  # noqa: BLE001
        print(f"[ERROR] Workflow failed: {exc}")

[INFO] Output dir: C:\Users\Work\github\leap_utilities\outputs\leap_results_dashboard\USA
[INFO] Using 6 LEAP workbook(s):
  - C:\Users\Work\github\leap_utilities\outputs\leap_results\demand_others_results_20_USA_Reference.xlsx
  - C:\Users\Work\github\leap_utilities\outputs\leap_results\demand_others_results_20_USA_Target.xlsx
  - C:\Users\Work\github\leap_utilities\outputs\leap_results\industry_results_20_USA_Reference.xlsx
  - C:\Users\Work\github\leap_utilities\outputs\leap_results\industry_results_20_USA_Target.xlsx
  - C:\Users\Work\github\leap_utilities\outputs\leap_results\transport_results_20_USA_Reference.xlsx
  - C:\Users\Work\github\leap_utilities\outputs\leap_results\transport_results_20_USA_Target.xlsx
[INFO] Wrote comparison_long: C:\Users\Work\github\leap_utilities\outputs\leap_results_dashboard\USA\comparison_long.csv
[WARN] Failed to render chart for Agriculture/Biodiesel: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed us